# 回测结果分析面板

这份 notebook 用于**结果分析**，默认读取导出的回测结果文件，重点看：

- 绩效摘要  
- 资产相关性矩阵
- 历史风险贡献度
- 净值与回撤  
- 年度收益  
- 月度收益热力图  
- 调仓次数与换手  
- 各资产平均权重与最后权重


In [ ]:

from pathlib import Path
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXPORT_DIR = Path("data/exports_risk_parity")

nav_df = pd.read_csv(EXPORT_DIR / "nav.csv", index_col=0, parse_dates=True)
weights_df = pd.read_csv(EXPORT_DIR / "weights.csv", index_col=0, parse_dates=True)
positions_df = pd.read_csv(EXPORT_DIR / "positions.csv", index_col=0, parse_dates=True)
target_weights_df = pd.read_csv(EXPORT_DIR / "target_weights.csv", index_col=0, parse_dates=True)
trades_df = pd.read_csv(EXPORT_DIR / "trades.csv")
rebalance_log_df = pd.read_csv(EXPORT_DIR / "rebalance_log.csv")
summary = pd.read_csv(EXPORT_DIR / "summary.csv", index_col=0)

asset_corr_matrix = pd.read_csv(
    EXPORT_DIR / "asset_corr_matrix.csv", 
    index_col=0
    )
risk_contribution_df = pd.read_csv(
    EXPORT_DIR / "risk_contribution_df.csv", 
    index_col=0, 
    parse_dates=True  # 核心：让 Pandas 把 Index 重新识别为时间格式
    )


plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

print("files loaded")
display(summary)


In [ ]:
# ========= 资产相关性矩阵 =========
# plt.figure(figsize=(8, 6))
# 使用 coolwarm 色系：红代表高度正相关，蓝代表负相关
sns.heatmap(
    asset_corr_matrix, 
    annot=True,        # 在格子里显示具体数值
    cmap='coolwarm',   # 冷暖色调
    vmin=-1, vmax=1,   # 相关性系数的理论极值
    center=0,          # 0 设为中间色（通常是白色或浅灰色）
    fmt='.2f',         # 保留两位小数
    linewidths=1,      # 格子之间的间距线
    square=True        # 保持格子是正方形
)
display(asset_corr_matrix)
plt.title('Asset Correlation Matrix', fontsize=14, pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# ========= 实际持有比例变化 =========
actual_weights_plot = weights_df.copy().fillna(0.0)

plt.figure(figsize=(14, 6))
plt.stackplot(
    actual_weights_plot.index,
    actual_weights_plot.T.values,
    labels=actual_weights_plot.columns
)
plt.title("Actual Portfolio Weights Over Time")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.ticker as ticker
# 创建画布
fig, ax = plt.subplots(figsize=(12, 6))

# 直接画折线图，而不是面积图
risk_contribution_df.plot.line(
    ax=ax, 
    linewidth=1.5, 
    alpha=0.8,
    colormap='Set1' # 使用对比度高的色系
)

# 1. 画一条 0 轴的黑色实线，用来捕捉负风险贡献
plt.axhline(0, color='black', linewidth=1.5, linestyle='-', zorder=1)

# 2. 画一条“目标风险平价线”的红色虚线
num_assets = len(risk_contribution_df.columns)
target_ratio = 1.0 / num_assets
plt.axhline(target_ratio, color='red', linewidth=2, linestyle='--', 
            label=f'Target RC ({target_ratio:.1%})', zorder=2)

plt.title('Historical Risk Contribution (Line Chart)', fontsize=14, pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Risk Contribution Ratio', fontsize=12)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))

# 调整图例
handles, labels = ax.get_legend_handles_labels()
plt.legend(handles, labels, loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

In [ ]:
# ========= 基础序列 =========
nav = nav_df["nav"].copy()
cash_weight = nav_df["cash"] / nav_df["nav"]
daily_ret = nav.pct_change().fillna(0.0)
drawdown = nav / nav.cummax() - 1.0

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# 1. 净值
axes[0].plot(nav.index, nav, label="NAV")
axes[0].set_title("Portfolio NAV")
axes[0].legend()

# 2. 回撤
axes[1].plot(drawdown.index, drawdown, label="Drawdown")
axes[1].set_title("Drawdown")
axes[1].legend()

# 3. 现金占比
axes[2].plot(cash_weight.index, cash_weight, label="Cash Weight")
axes[2].set_title("Cash Weight")
axes[2].legend()

plt.tight_layout()
plt.show()

print("平均现金占比：", round(cash_weight.mean(), 4))
print("最大现金占比：", round(cash_weight.max(), 4))
print("最近现金占比：", round(cash_weight.iloc[-1], 4))

In [ ]:

# ========= 年度收益 =========
year_end_nav = nav.resample("Y").last()
annual_returns = year_end_nav.pct_change()
annual_returns.index = annual_returns.index.year

annual_returns_df = annual_returns.to_frame("annual_return")
display(annual_returns_df)

plt.figure(figsize=(10, 4))
plt.bar(annual_returns_df.index.astype(str), annual_returns_df["annual_return"])
plt.title("Annual Returns")
plt.show()


In [ ]:

# ========= 月度收益热力图（表格版） =========
month_end_nav = nav.resample("M").last()
monthly_returns = month_end_nav.pct_change()
monthly_df = monthly_returns.to_frame("monthly_return")
monthly_df["year"] = monthly_df.index.year
monthly_df["month"] = monthly_df.index.month

heatmap_table = monthly_df.pivot(index="year", columns="month", values="monthly_return")
display(heatmap_table.style.format("{:.2%}"))
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(heatmap_table.values, aspect="auto")

ax.set_xticks(range(len(heatmap_table.columns)))
ax.set_xticklabels(heatmap_table.columns)

ax.set_yticks(range(len(heatmap_table.index)))
ax.set_yticklabels(heatmap_table.index)

ax.set_title("Monthly Return Heatmap")
ax.set_xlabel("Month")
ax.set_ylabel("Year")

for i in range(heatmap_table.shape[0]):
    for j in range(heatmap_table.shape[1]):
        val = heatmap_table.iloc[i, j]
        if pd.notna(val):
            ax.text(j, i, f"{val:.1%}", ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:

# ========= 调仓统计 =========
if len(rebalance_log_df) > 0:
    rebalance_log_df["trade_date"] = pd.to_datetime(rebalance_log_df["trade_date"])
    rebalance_log_df["year"] = rebalance_log_df["trade_date"].dt.year

    print("调仓日志：")
    display(rebalance_log_df.tail(20))

    yearly_rebalance_count = rebalance_log_df.groupby("year").size().rename("rebalance_count")
    yearly_turnover = rebalance_log_df.groupby("year")["turnover"].sum().rename("turnover_sum")

    display(pd.concat([yearly_rebalance_count, yearly_turnover], axis=1))
else:
    print("没有调仓记录")


In [ ]:

# ========= 交易统计 =========
if len(trades_df) > 0:
    trade_summary = trades_df.groupby(["ts_code", "side"])["trade_value"].agg(["count", "sum"])
    display(trade_summary)

    fee_summary = trades_df.groupby("side")["cost"].sum().rename("total_cost")
    display(fee_summary)
else:
    print("没有交易记录")


In [ ]:

# ========= 权重分析 =========
print("最后一期实际权重：")
display(weights_df.tail(1).T.rename(columns={weights_df.index[-1]: "last_weight"}).sort_values("last_weight", ascending=False))

print("样本期平均实际权重：")
avg_weight = weights_df.mean().sort_values(ascending=False).to_frame("avg_weight")
display(avg_weight)

plt.figure(figsize=(10, 4))
avg_weight["avg_weight"].plot(kind="bar")
plt.title("Average Actual Weights")
plt.show()


In [ ]:

# ========= 最近持仓与目标权重对比 =========
if len(weights_df) > 0 and len(target_weights_df) > 0:
    last_actual = weights_df.tail(1).T.rename(columns={weights_df.index[-1]: "actual_weight"})
    last_target = target_weights_df.tail(1).T.rename(columns={target_weights_df.index[-1]: "target_weight"})
    compare = last_actual.join(last_target, how="outer").fillna(0.0)
    compare["diff"] = compare["actual_weight"] - compare["target_weight"]
    display(compare.sort_values("target_weight", ascending=False))


In [ ]:

# ========= 核心结论快照 =========
snapshot = {
    "summary": summary,
    "last_nav": nav.iloc[-1],
    "max_drawdown": drawdown.min(),
    "last_positions": positions_df.tail(1).T if len(positions_df) > 0 else None,
    "last_weights": weights_df.tail(1).T if len(weights_df) > 0 else None,
    "rebalance_count": len(rebalance_log_df),
    "trade_count": len(trades_df),
}

snapshot["summary"]
